# SQL Injection Prevention

## Overview
SQL injection is the most common and highest-impact database vulnerability. It occurs when user-supplied input is concatenated into SQL and interpreted as commands rather than data. The defence is simple and absolute: **always use parameterised queries**.

```python
# NEVER — string formatting
sql = f"SELECT * FROM patients WHERE id = '{user_input}'"

# ALWAYS — parameterised
conn.execute("SELECT * FROM patients WHERE id = ?", (user_input,))
```

For dynamic identifiers (table/column names) that cannot be parameterised: use a hardcoded allowlist, then `quote_ident()` / `format('%I', name)` in PostgreSQL.

---

In [ ]:
import sqlite3, hashlib, secrets, base64, hmac
from datetime import datetime

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE users (
    user_id TEXT PRIMARY KEY, username TEXT NOT NULL UNIQUE,
    role_name TEXT NOT NULL, department TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY, full_name TEXT NOT NULL,
    dob TEXT NOT NULL, province TEXT NOT NULL, assigned_clinician TEXT
);
CREATE TABLE patient_records (
    record_id TEXT PRIMARY KEY, patient_id TEXT NOT NULL,
    record_type TEXT NOT NULL, content TEXT NOT NULL,
    sensitivity TEXT DEFAULT 'normal', created_by TEXT NOT NULL, created_at TEXT
);
CREATE TABLE financial_accounts (
    account_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL,
    account_type TEXT NOT NULL, balance_enc TEXT, ssn_hash TEXT, created_at TEXT
);
CREATE TABLE audit_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT, event_time TEXT DEFAULT (datetime('now')),
    username TEXT NOT NULL, action TEXT NOT NULL, table_name TEXT,
    record_id TEXT, old_value TEXT, new_value TEXT, ip_address TEXT, success INTEGER DEFAULT 1
);
CREATE TABLE role_permissions (
    role_name TEXT NOT NULL, table_name TEXT NOT NULL,
    can_select INTEGER DEFAULT 0, can_insert INTEGER DEFAULT 0,
    can_update INTEGER DEFAULT 0, can_delete INTEGER DEFAULT 0,
    row_filter TEXT, PRIMARY KEY (role_name, table_name)
);
""")

conn.executemany("INSERT INTO users VALUES (?,?,?,?,?)", [
    ('U001','dr.lee',    'clinician','Cardiology',    1),
    ('U002','dr.pham',   'clinician','Endocrinology',1),
    ('U003','analyst1',  'analyst',  'Reporting',    1),
    ('U004','admin1',    'admin',    'IT',           1),
    ('U005','auditor1',  'auditor',  'Compliance',   1),
    ('U006','patient_p001','patient',None,            1),
])
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", [
    ('P001','Alice Ngata',  '1980-03-15','NB','dr.lee'),
    ('P002','Bob Chen',     '1972-07-22','ON','dr.pham'),
    ('P003','Fatima Rashid','1986-11-05','BC','dr.lee'),
    ('P004','James MacLeod','1963-01-30','NS','dr.pham'),
    ('P005','Mei-Ling Tran','1966-08-19','QC','dr.lee'),
])
conn.executemany("INSERT INTO patient_records VALUES (?,?,?,?,?,?,?)", [
    ('R001','P001','diagnosis',  'Hypertension, Stage 2',      'normal',    'dr.lee', '2024-01-15'),
    ('R002','P001','prescription','Lisinopril 10mg daily',     'normal',    'dr.lee', '2024-01-15'),
    ('R003','P002','diagnosis',  'Type 2 Diabetes',            'normal',    'dr.pham','2024-01-10'),
    ('R004','P002','lab',        'HbA1c 8.4%',                 'normal',    'dr.pham','2024-01-10'),
    ('R005','P003','diagnosis',  'Asthma, moderate persistent','sensitive', 'dr.lee', '2024-02-01'),
    ('R006','P004','diagnosis',  'CKD Stage 3, T2DM',          'sensitive', 'dr.pham','2024-02-15'),
    ('R007','P005','prescription','Insulin glargine 20u',      'restricted','dr.lee', '2024-03-01'),
])
salt = secrets.token_hex(8)
def hash_ssn(ssn): return hashlib.sha256((salt+ssn).encode()).hexdigest()
conn.executemany("INSERT INTO financial_accounts VALUES (?,?,?,?,?,datetime('now'))", [
    ('ACC001','P001','chequing','[encrypted:AES256]',hash_ssn('123-45-6789')),
    ('ACC002','P002','savings', '[encrypted:AES256]',hash_ssn('987-65-4321')),
    ('ACC003','P003','chequing','[encrypted:AES256]',hash_ssn('456-78-9012')),
])
conn.executemany("INSERT INTO role_permissions VALUES (?,?,?,?,?,?,?)", [
    ('clinician','patients',       1,0,1,0,'assigned_clinician = current_user'),
    ('clinician','patient_records',1,1,1,0,'patient_id IN (SELECT patient_id FROM patients WHERE assigned_clinician = current_user)'),
    ('analyst',  'patients',       1,0,0,0,'province IS NOT NULL'),
    ('analyst',  'patient_records',1,0,0,0,"sensitivity = 'normal'"),
    ('admin',    'patients',       1,1,1,1,None),
    ('admin',    'patient_records',1,1,1,1,None),
    ('auditor',  'audit_log',      1,0,0,0,None),
    ('auditor',  'patients',       1,0,0,0,None),
    ('patient',  'patients',       1,0,0,0,'patient_id = current_patient_id()'),
    ('patient',  'patient_records',1,0,0,0,'patient_id = current_patient_id()'),
])
conn.executemany(
    "INSERT INTO audit_log (username,action,table_name,record_id,old_value,new_value,ip_address,success) VALUES (?,?,?,?,?,?,?,?)", [
    ('dr.lee',  'SELECT','patient_records','R001',None,None,'10.0.1.5',1),
    ('dr.lee',  'UPDATE','patient_records','R002','{"content":"Lisinopril 5mg"}','{"content":"Lisinopril 10mg"}','10.0.1.5',1),
    ('analyst1','SELECT','patients',       None,  None,None,'10.0.2.8',1),
    ('analyst1','EXPORT','patient_records',None,  None,None,'10.0.2.8',1),
    ('unknown', 'LOGIN', None,             None,  None,None,'203.0.113.9',0),
    ('dr.pham', 'SELECT','patient_records','R005',None,None,'10.0.1.6',0),
    ('admin1',  'DELETE','patient_records','R007',None,None,'10.0.1.1',1),
])
conn.commit()
print("Dataset loaded:")
for t in ['users','patients','patient_records','financial_accounts','audit_log','role_permissions']:
    print(f"  {t}: {conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]} rows")

print("=== SQL injection: the vulnerability ===")
print()
def vulnerable(user_input):
    return f"SELECT * FROM users WHERE username = '{user_input}'"

attacks = [
    ("Normal",           "dr.lee",
     "Works as intended"),
    ("Always-true",      "' OR '1'='1",
     "Returns ALL users — authentication bypass"),
    ("Comment injection","admin'--",
     "Comments out the rest of the query"),
    ("UNION injection",  "' UNION SELECT user_id,username,'hacked','IT',1 FROM users--",
     "Appends attacker-controlled rows"),
    ("DROP TABLE",       "'; DROP TABLE patients;--",
     "Destroys the patients table"),
]
for label, attack, effect in attacks:
    sql = vulnerable(attack)
    print(f"  [{label}]")
    print(f"    Input: {repr(attack)}")
    print(f"    SQL:   {sql[:90]}")
    print(f"    Risk:  {effect}")
    print()


In [ ]:

print("=== Parameterised queries: the correct approach ===")
print()
print("Parameterised queries separate SQL structure from data.")
print("The driver escapes data at the binary protocol level — it can never become SQL.")
print()

searches = [
    ("Normal input",       "dr.lee"),
    ("Always-true attack", "' OR '1'='1"),
    ("Comment attack",     "admin'--"),
]
print("sqlite3 with ? placeholder (injection attempts return 0 rows):")
for label, user_input in searches:
    rows = conn.execute(
        "SELECT username, role_name FROM users WHERE username = ?", (user_input,)
    ).fetchall()
    print(f"  {label:<26s}  input={repr(user_input):<22s}  found={len(rows)}")

print()
drivers = [
    ("sqlite3",    "?",    "conn.execute('SELECT * FROM t WHERE id = ?', (id,))"),
    ("psycopg2",   "%s",   "cur.execute('SELECT * FROM t WHERE id = %s', (id,))"),
    ("asyncpg",    "$1",   "await conn.fetch('SELECT * FROM t WHERE id = $1', id)"),
    ("SQLAlchemy", ":name","session.execute(text('... WHERE id = :id'), {'id': id})"),
    ("dbt",        "{{ }}", "{{ ref() }} and {{ var() }} — never string-format SQL"),
]
print(f"  {'Driver':<12s}  {'Placeholder':<12s}  Example")
print("  " + "-"*72)
for d, p, ex in drivers:
    print(f"  {d:<12s}  {p:<12s}  {ex}")

print()
print("=== Safe dynamic SQL with allowlists ===")
ALLOWED_COLS   = {'patient_id','full_name','province','dob'}
ALLOWED_TABLES = {'patients','patient_records'}
ALLOWED_DIRS   = {'ASC','DESC'}

def safe_query(table, col, direction, conn):
    if table not in ALLOWED_TABLES: raise ValueError(f"Table not allowed: {table}")
    if col not in ALLOWED_COLS:     raise ValueError(f"Column not allowed: {col}")
    if direction.upper() not in ALLOWED_DIRS: raise ValueError(f"Direction not allowed: {direction}")
    return conn.execute(
        f"SELECT patient_id, full_name FROM {table} ORDER BY {col} {direction.upper()}"
    ).fetchall()

test_cases = [
    ("patients","full_name","ASC",  "valid"),
    ("users",   "username", "ASC",  "invalid table"),
    ("patients","password", "ASC",  "invalid column"),
    ("patients","province", "UNION SELECT","invalid direction"),
]
print()
print("Allowlist validation (live):")
for table, col, direction, expected in test_cases:
    try:
        rows = safe_query(table, col, direction, conn)
        print(f"  [{expected}]   {table}.{col} {direction} → {len(rows)} rows")
    except ValueError as e:
        print(f"  [blocked ✓]  {table}.{col} {direction} → {e}")

print()
print("PostgreSQL safe dynamic identifiers (quote_ident / format):")
print("""
  -- %I = quote_ident() — safely quotes table/column names
  -- %L = quote_literal() — safely quotes string values
  -- USING clause = parameterised values in EXECUTE

  EXECUTE format('SELECT * FROM %I WHERE province = $1', table_name) USING province;
  -- table_name is safely quoted; province is a parameter
""")


---
## Common Pitfalls

**1. Using string formatting to build SQL**
`f"SELECT * FROM patients WHERE name = '{user_input}'"` is vulnerable regardless of input source. There is no scenario where string-formatting user input into SQL is safe. Parameterise, always.

**2. Sanitising input with replace() or regex instead of parameterising**
Attempts to strip quotes or escape characters are incomplete and bypassable. Database drivers escape at the binary protocol level in ways string manipulation cannot replicate.

**3. Assuming ORM usage means injection is impossible**
SQLAlchemy model queries are safe. But `session.execute(text(f'... WHERE id = {user_id}'))` is just as vulnerable. Always parameterise when dropping to raw SQL: `session.execute(text('... WHERE id = :id'), {'id': user_id})`.

**4. Allowlists stored in the database**
If permitted column names come from a table that itself can be modified via SQL injection, the attacker adds their chosen column to the allowlist. Allowlists for dynamic SQL must be hardcoded in application code.

**5. Second-order SQL injection via stored data**
Data stored from a prior injection that wasn't exploited at insertion time can be exploited later when used to build a dynamic query. Parameterise at both insert AND every subsequent use.

**6. Not using quote_ident() for dynamic identifiers in PL/pgSQL**
`EXECUTE 'SELECT * FROM ' || table_name` is vulnerable. Use `format('SELECT * FROM %I', table_name)` where `%I` calls `quote_ident()` — safely double-quotes the identifier.

---
*sql_methods_library - Samantha McGarrigle*